# Terra Signal: decisão de abordagem e estrutura de incrementalidade (Grupo 1)

Transforma o risco do modelo em decisão de negócio e deixa pronta a estrutura para medir o impacto real da ação.

Cada pergunta tem três células: **pergunta** → **código** → **resposta**.

- **Ação proposta:** duas camadas. Na **reativa**, toda ligação de cancelamento recebe a oferta, sem depender do modelo. Na **proativa**, a TS liga para os clientes do topo do ranking. A oferta é a mesma nas duas: *migrar para o contrato anual e ganhar suporte técnico grátis por 12 meses*.
- **Este notebook trata só da camada proativa**, que é onde o modelo decide quem abordar.
- **O modelo estima risco; a regra de decisão transforma o risco em lucro.** Liga-se para o cliente quando o lucro esperado da ligação é positivo.
- **As taxas de comportamento são premissas.** Não há histórico de contatos nem de ofertas, então elas entram como três cenários ilustrativos. O piloto da pergunta 5 é o que mede essas taxas de verdade.
- Roda com pandas, numpy, scipy e scikit-learn, sem Spark.

## Conta do lucro por cliente abordado

```
lucro_i = p_i × Δa × (V_i − D)  −  (1 − p_i) × (u × D + w × V_i)  −  c
```

| Termo | O que é | Valor |
|---|---|---|
| `p_i` | Probabilidade de churn em 12 meses, sem ação | Modelo |
| `V_i` | Receita de um cliente mantido | 12 × `MonthlyCharges` do cliente |
| `D` | Custo da oferta por cliente que aceita | Até R$ 60 (R$ 5 de suporte × 12 meses) |
| `c` | Custo de uma ligação | R$ 10 (R$ 5 mil por atendente/mês ÷ 525 abordagens) |
| `Δa` | Fração dos churners abordados que a ligação salva, além do que a camada reativa salvaria | Cenário |
| `w` | Fração dos não churners abordados que cancelam por terem sido lembrados ("cão adormecido") | Cenário |
| `u` | Fração dos não churners que aceitam a oferta sem precisar | 30% |

| Cenário | `Δa` | `w` |
|---|---|---|
| Otimista | 30% | 0,5% |
| Base | 20% | 1% |
| Pessimista | 10% | 3% |

## Setup

In [1]:
import math
import warnings

import numpy as np
import pandas as pd
from scipy.stats import chi2, norm
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GridSearchCV, RepeatedStratifiedKFold, StratifiedKFold

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)

DATA_DIR = "../data/raw/"  # este notebook fica em notebooks/; os CSVs ficam em data/raw/
hist_raw = pd.read_csv(DATA_DIR + "history.csv")
inf_raw = pd.read_csv(DATA_DIR + "inference.csv")
print("history:", hist_raw.shape, "| inference:", inf_raw.shape)

history: (5634, 23) | inference: (1409, 22)


In [2]:
# Premissas de negócio
MESES_V = 12        # V_i = 12 x MonthlyCharges_i, a duração do contrato anual oferecido
D = 60.0            # custo máximo da oferta por cliente que aceita
C_LIGACAO = 10.0    # custo por ligação, a revisar
LIGACOES_MES = 525  # abordagens por atendente por mês
U = 0.30            # aceite da oferta entre quem não ia sair
C_FINAL = 1         # escolhido na pergunta 8 do notebook 09
CENARIOS = {
    "otimista": {"da": 0.30, "w": 0.005},
    "base": {"da": 0.20, "w": 0.01},
    "pessimista": {"da": 0.10, "w": 0.03},
}

In [3]:
def prep(d):
    """Regras do notebook 06: recupera tenure. TotalCharges só serve para isso e não entra no modelo."""
    d = d.copy()
    d["tenure_num"] = pd.to_numeric(d["tenure"], errors="coerce")
    d["tenure_num"] = d["tenure_num"].fillna((d["TotalCharges"] / d["MonthlyCharges"]).round())
    return d


hist = prep(hist_raw)
hist = hist[hist["tenure_num"].notna()].reset_index(drop=True)  # 1 linha sem tenure e sem TotalCharges
inf = prep(inf_raw)
y = (hist["Churn"] == "Yes").astype(int)
print("Clientes no treino:", len(hist), "| churners:", y.sum(), f"({y.mean():.1%})")

Clientes no treino: 5633 | churners: 1503 (26.7%)


In [4]:
def melhor_corte(x, y, min_folha):
    """Corte binário de maior qui-quadrado, com p-valor corrigido por Bonferroni pelo nº de cortes testados."""
    tab = pd.DataFrame({"x": x, "y": y}).groupby("x")["y"].agg(["size", "sum"]).sort_index()
    n_esq, c_esq = tab["size"].cumsum().values, tab["sum"].cumsum().values
    n, c = n_esq[-1], c_esq[-1]
    validos = (n_esq >= min_folha) & (n - n_esq >= min_folha)
    if not validos.any():
        return None
    a, b = c_esq, n_esq - c_esq
    e, f = c - c_esq, (n - n_esq) - (c - c_esq)
    den = (a + b) * (e + f) * (a + e) * (b + f)
    with np.errstate(divide="ignore", invalid="ignore"):
        q = np.where(den > 0, n * (a * f - b * e) ** 2 / den, 0.0)
    q = np.where(validos, q, -1.0)
    i = int(np.argmax(q))
    return float(tab.index[i]), float(q[i]), min(1.0, chi2.sf(q[i], 1) * int(validos.sum()))


def aid(x, y, min_folha=0.05, alfa=0.05, max_faixas=None):
    """AID do notebook 07: divide sempre a faixa com maior qui-quadrado."""
    x, y = np.asarray(x, float), np.asarray(y)
    tam = int(np.ceil(min_folha * len(x)))
    nos, cortes = [np.ones(len(x), bool)], []
    while not max_faixas or len(nos) < max_faixas:
        candidatos = []
        for k, m in enumerate(nos):
            r = melhor_corte(x[m], y[m], tam)
            if r and r[2] < alfa:
                candidatos.append((r[1], k, r))
        if not candidatos:
            break
        _, k, r = max(candidatos, key=lambda z: z[0])
        m = nos.pop(k)
        nos += [m & (x <= r[0]), m & (x > r[0])]
        cortes.append(r[0])
    return sorted(cortes)


DUMMIES_CATEG = {
    "gender": ["Female"],
    "SeniorCitizen": [1],
    "Partner": ["Yes"],
    "Dependents": ["Yes"],
    "MultipleLines": ["Yes", "No phone service"],
    "InternetService": ["DSL", "No"],
    "OnlineSecurity": ["Yes"],
    "OnlineBackup": ["Yes"],
    "DeviceProtection": ["Yes"],
    "TechSupport": ["Yes"],
    "StreamingTV": ["Yes"],
    "StreamingMovies": ["Yes"],
    "Contract": ["One year", "Two year"],
    "PaperlessBilling": ["No"],
    "PaymentMethod": ["Mailed check", "Credit card (automatic)", "Bank transfer (automatic)"],
}


def dummies_faixas(x, cortes, ref):
    """Uma dummy por faixa de tenure, exceto a faixa de referência `ref`."""
    f = np.digitize(x, cortes, right=True)
    limites = [-np.inf] + list(cortes) + [np.inf]
    return pd.DataFrame(
        {f"tenure=({limites[i]:g}, {limites[i + 1]:g}]": (f == i).astype(int) for i in range(len(limites) - 1) if i != ref},
        index=x.index,
    )


def montar_dummies(d, cortes, ref):
    cat = pd.DataFrame(
        {f"{col}={val}": (d[col] == val).astype(int) for col, vals in DUMMIES_CATEG.items() for val in vals},
        index=d.index,
    )
    return pd.concat([cat, dummies_faixas(d["tenure_num"], cortes, ref)], axis=1)


def faixa_referencia(x, cortes):
    return int(pd.Series(np.digitize(x, cortes, right=True)).value_counts().idxmax())

### 1. Qual é a probabilidade de churn de cada cliente, calculada sem que ele esteja no treino?

In [5]:
CV = RepeatedStratifiedKFold(n_splits=5, n_repeats=4, random_state=0)  # mesmos folds do notebook 09
INTERNO = StratifiedKFold(3, shuffle=True, random_state=0)
previsoes = np.full((4, len(hist)), np.nan)
c_escolhidos = []
for i, (tr, te) in enumerate(CV.split(hist, y)):
    cortes = aid(hist["tenure_num"].iloc[tr], y.iloc[tr], max_faixas=6)
    ref = faixa_referencia(hist["tenure_num"].iloc[tr], cortes)
    g = GridSearchCV(LogisticRegression(max_iter=3000), {"C": [0.01, 0.1, 1, 10]}, scoring="roc_auc", cv=INTERNO)
    g.fit(montar_dummies(hist.iloc[tr], cortes, ref), y.iloc[tr])
    c_escolhidos.append(g.best_params_["C"])
    previsoes[i // 5, te] = g.predict_proba(montar_dummies(hist.iloc[te], cortes, ref))[:, 1]

P = previsoes.mean(axis=0)  # média das 4 repetições
print("C escolhido nos 20 folds:", pd.Series(c_escolhidos).value_counts().to_dict())
print("Clientes sem previsão:", int(np.isnan(previsoes).sum()))
print("AUC por repetição:", [round(roc_auc_score(y, previsoes[r]), 4) for r in range(4)])
print(f"AUC da média das repetições: {roc_auc_score(y, P):.4f}")

C escolhido nos 20 folds: {1: 17, 10: 3}
Clientes sem previsão: 0
AUC por repetição: [0.8429, 0.8442, 0.8456, 0.8456]
AUC da média das repetições: 0.8459


**Resposta:** os 5.633 clientes do `history` têm uma probabilidade calculada por modelos que não os viram no treino, com o mesmo procedimento do notebook 11. A AUC da média das 4 repetições é de **0,846**. Essas probabilidades são a base das perguntas 2, 3 e 5. Por isso, nenhuma comparação deste notebook é feita com o modelo avaliado nos próprios dados de treino.

### 2. Com o mesmo número de ligações, o modelo encontra mais churners do que a regra "mensal sem suporte técnico"?

A regra é a alternativa que dispensa modelo: ligar para quem tem contrato mensal e não tem suporte técnico. Como a regra não ordena os clientes dentro do grupo, uma lista menor que o grupo é tratada como sorteio dentro dele.

In [6]:
regra = ((hist["Contract"] == "Month-to-month") & (hist["TechSupport"] == "No")).values
V = MESES_V * hist["MonthlyCharges"].values
yv = y.values
ordem = np.argsort(-P)
n_regra = int(regra.sum())
print(f"Grupo da regra: {n_regra} clientes ({regra.mean():.1%} da base), churn de {yv[regra].mean():.1%}, "
      f"com {yv[regra].sum() / yv.sum():.1%} dos churners\n")

linhas = []
for k in [250, 500, 1000, n_regra]:
    topo = ordem[:k]
    linhas.append({
        "ligacoes": k,
        "churners_modelo": int(yv[topo].sum()),
        "churners_regra": round(k * yv[regra].mean()),
        "receita_em_risco_modelo": (yv[topo] * V[topo]).sum(),
        "receita_em_risco_regra": k * (yv[regra] * V[regra]).mean(),
        "clientes_fora_da_regra_no_topo": int((~regra[topo]).sum()),
    })
comp = pd.DataFrame(linhas)
comp["ganho_churners"] = comp["churners_modelo"] / comp["churners_regra"] - 1
comp["ganho_receita"] = comp["receita_em_risco_modelo"] / comp["receita_em_risco_regra"] - 1
print(comp.round(3).to_string(index=False))

Grupo da regra: 2154 clientes (38.2% da base), churn de 50.4%, com 72.2% dos churners

 ligacoes  churners_modelo  churners_regra  receita_em_risco_modelo  receita_em_risco_regra  clientes_fora_da_regra_no_topo  ganho_churners  ganho_receita
      250              212             126                 208708.2              114869.290                               3           0.683          0.817
      500              391             252                 379772.4              229738.579                              16           0.552          0.653
     1000              699             504                 670656.0              459477.159                              61           0.387          0.460
     2154             1161            1085                1084056.6              989713.800                             395           0.070          0.095


**Resposta:** sim, e a vantagem é maior quando a lista é curta.

| Ligações | Churners (modelo) | Churners (regra) | Ganho em churners | Ganho em receita em risco |
|---|---|---|---|---|
| 250 | 212 | 126 | **+68%** | +82% |
| 500 | 391 | 252 | **+55%** | +65% |
| 1.000 | 699 | 504 | **+39%** | +46% |
| 2.154 (grupo inteiro da regra) | 1.161 | 1.085 | **+7%** | +10% |

- **A regra é forte:** ela aponta 38,2% da base, com churn de 50,4% e 72,2% de todos os churners.
- **O modelo ganha por ordenar:** dentro do grupo da regra, escolher quem ligar é um sorteio com 50% de acerto. Nos 250 primeiros do ranking, 85% são churners.
- **Com o grupo inteiro, o ganho é pequeno:** são 7% a mais de churners, e 395 dos clientes de maior risco ficam fora da regra.
- **A comparação favorece a regra:** ela foi escolhida olhando o `history` inteiro, enquanto o modelo foi avaliado fora da amostra.

### 3. Para quem vale ligar em cada cenário, e quanto isso rende no `history`?

Liga-se para o cliente quando `lucro_i` é positivo. O **lucro esperado** usa a probabilidade do modelo. O **lucro com o churn observado** troca `p_i` pelo churn real de cada cliente, o que funciona como um teste com dados que o modelo não viu, já que as probabilidades são fora da amostra. As duas versões dependem das premissas de `Δa` e `w`.

In [7]:
def lucro_cliente(p, V, da, w):
    return p * da * (V - D) - (1 - p) * (U * D + w * V) - C_LIGACAO


linhas = []
for nome, s in CENARIOS.items():
    alvo = lucro_cliente(P, V, **s) > 0
    for estrategia, m in [("modelo", alvo), ("regra", regra), ("todos", np.ones(len(P), bool))]:
        linhas.append({
            "cenario": nome, "estrategia": estrategia, "ligacoes": int(m.sum()), "pct_base": m.mean(),
            "menor_p_abordado": P[m].min() if estrategia == "modelo" else np.nan,
            "churn_dos_abordados": yv[m].mean(),
            "lucro_esperado": lucro_cliente(P[m], V[m], **s).sum(),
            "lucro_churn_observado": lucro_cliente(yv[m], V[m], **s).sum(),
            "atendentes_1_mes": math.ceil(m.sum() / LIGACOES_MES),
        })
dec = pd.DataFrame(linhas)
print(dec.round(3).to_string(index=False))

   cenario estrategia  ligacoes  pct_base  menor_p_abordado  churn_dos_abordados  lucro_esperado  lucro_churn_observado  atendentes_1_mes
  otimista     modelo      3011     0.535             0.085                0.437      284493.089             280913.119                 6
  otimista      regra      2154     0.382               NaN                0.504      229801.401             232114.494                 5
  otimista      todos      5633     1.000               NaN                0.267      230488.389             230862.451                11
      base     modelo      2524     0.448             0.150                0.490      162280.487             161266.982                 5
      base      regra      2154     0.382               NaN                0.504      133524.716             135165.468                 5
      base      todos      5633     1.000               NaN                0.267       89956.512              90210.262                11
pessimista     modelo      1452   

**Resposta:** o modelo dá mais lucro do que a regra nos três cenários, e a vantagem cresce quando o cenário piora.

| Cenário | Ligações do modelo | Menor `p` abordado | Lucro do modelo | Lucro da regra | Lucro de ligar para todos |
|---|---|---|---|---|---|
| Otimista | 3.011 (53%) | 0,085 | **R$ 281 mil** | R$ 232 mil | R$ 231 mil |
| Base | 2.524 (45%) | 0,150 | **R$ 161 mil** | R$ 135 mil | R$ 90 mil |
| Pessimista | 1.452 (26%) | 0,368 | **R$ 42 mil** | R$ 25 mil | **−R$ 96 mil** |

*Lucro com o churn observado, em receita de 12 meses, para os 5.633 clientes do `history`.*

- **O modelo supera a regra em 21%, 19% e 69%**, do cenário otimista ao pessimista. Quanto mais caro é errar, mais vale escolher bem quem ligar.
- **Ligar para todos dá prejuízo no cenário pessimista:** sem escolher, os clientes "acordados" e as ofertas dadas sem necessidade custam mais do que os churners salvos rendem.
- **O corte é bem menor que 0,5 em todos os cenários.** Ele muda de cliente para cliente, porque depende da mensalidade: um cliente que paga mais justifica a ligação com um risco menor.
- **O lucro esperado e o lucro com o churn observado quase coincidem**, porque as probabilidades estão calibradas.

### 4. Quantos clientes do `inference` a TS abordaria em cada cenário, e com quantos atendentes?

In [8]:
cortes_final = aid(hist["tenure_num"], y, max_faixas=6)
ref_final = faixa_referencia(hist["tenure_num"], cortes_final)
modelo = LogisticRegression(C=C_FINAL, max_iter=3000).fit(montar_dummies(hist, cortes_final, ref_final), y)
p_inf = modelo.predict_proba(montar_dummies(inf, cortes_final, ref_final))[:, 1]
V_inf = MESES_V * inf["MonthlyCharges"].values

linhas = []
for nome, s in CENARIOS.items():
    l = lucro_cliente(p_inf, V_inf, **s)
    alvo = l > 0
    linhas.append({"cenario": nome, "ligacoes": int(alvo.sum()), "pct_inference": alvo.mean(),
                   "churners_esperados_no_alvo": p_inf[alvo].sum(), "lucro_esperado": l[alvo].sum(),
                   "atendentes_1_mes": math.ceil(alvo.sum() / LIGACOES_MES)})
print(pd.DataFrame(linhas).round(3).to_string(index=False))

   cenario  ligacoes  pct_inference  churners_esperados_no_alvo  lucro_esperado  atendentes_1_mes
  otimista       742          0.527                     329.845       71110.467                 2
      base       634          0.450                     311.161       40658.814                 2
pessimista       364          0.258                     223.593       10671.560                 1

**Resposta:** no cenário base, a TS ligaria para **634 dos 1.409 clientes (45%)**, com **2 atendentes durante um mês**.

| Cenário | Ligações | Fatia do `inference` | Atendentes por 1 mês | Lucro esperado |
|---|---|---|---|---|
| Otimista | 742 | 52,7% | 2 | R$ 71 mil |
| Base | 634 | 45,0% | 2 | R$ 41 mil |
| Pessimista | 364 | 25,8% | 1 | R$ 11 mil |

No cenário base, a lista de 634 clientes concentra cerca de 311 dos churners esperados. Se a base ativa da TS for maior, a lista e o time crescem na mesma proporção: um atendente para cada 525 ligações por mês.

### 5. De quantos clientes o piloto precisa para medir o efeito da ligação?

O piloto sorteia os clientes elegíveis (lucro esperado positivo) em dois grupos do mesmo tamanho:

| Grupo | Ação | Para que serve |
|---|---|---|
| **A** | Ligação com a oferta | Mostra o churn com a ação |
| **B** | Sem ligação, só a camada reativa | Mostra o churn sem a ação |

A diferença de churn entre B e A é o efeito líquido da ligação: churners salvos menos clientes "acordados". O cálculo supõe 80% de poder e 5% de significância bilateral, com o churn medido em 12 meses. Supõe também que a base ativa da TS é maior que o `inference`, de onde os grupos são sorteados.

In [9]:
def clientes_por_grupo(p0, p1, alfa=0.05, poder=0.80):
    z = norm.ppf(1 - alfa / 2) + norm.ppf(poder)
    return math.ceil(z ** 2 * (p0 * (1 - p0) + p1 * (1 - p1)) / (p0 - p1) ** 2)


linhas = []
for nome, s in CENARIOS.items():
    alvo = lucro_cliente(P, V, **s) > 0
    p0 = P[alvo].mean()
    efeito = (P[alvo] * s["da"] - (1 - P[alvo]) * s["w"]).mean()
    n = clientes_por_grupo(p0, p0 - efeito)
    linhas.append({"cenario": nome, "churn_sem_acao": p0, "churn_com_ligacao": p0 - efeito, "queda_pp": 100 * efeito,
                   "clientes_grupo_A": n, "clientes_grupo_B": n, "total_piloto": 2 * n,
                   "atendentes_grupo_A_1_mes": math.ceil(n / LIGACOES_MES)})
piloto = pd.DataFrame(linhas)
print(piloto.round(3).to_string(index=False))

   cenario  churn_sem_acao  churn_com_ligacao  queda_pp  clientes_grupo_A  clientes_grupo_B  total_piloto  atendentes_grupo_A_1_mes
  otimista           0.441              0.311    12.942               216               216           432                         1
      base           0.492              0.399     9.341               441               441           882                         1
pessimista           0.616              0.566     5.012              1507              1507          3014                         3


**Resposta:** no cenário base, o piloto precisa de **441 clientes em cada grupo, ou 882 no total**. Só o grupo A recebe ligação, então basta um atendente durante um mês.

| Cenário | Churn sem ação | Churn com ligação | Queda | Clientes por grupo | Total no piloto |
|---|---|---|---|---|---|
| Otimista | 44,1% | 31,1% | 12,9 p.p. | 216 | 432 |
| Base | 49,2% | 39,9% | 9,3 p.p. | **441** | **882** |
| Pessimista | 61,6% | 56,6% | 5,0 p.p. | 1.507 | 3.014 |

- **Quanto menor o efeito, maior o piloto:** no cenário pessimista, são necessários quase 3,5 vezes mais clientes.
- **O piloto não cabe só no `inference`:** no cenário base, há 634 clientes elegíveis ali, menos que os 882 necessários. Por isso a premissa de uma base ativa maior.
- **As janelas:** aos **90 dias**, o piloto serve de alerta. Se os cancelamentos logo depois da ligação forem maiores no grupo A ou o aceite ficar muito abaixo do esperado, a ação é revista. A **decisão** sai aos **12 meses**, com a calculadora da pergunta 6.

### 6. Com os resultados do piloto em mãos, como calcular o impacto?

A calculadora abaixo recebe o que o piloto mede e devolve o impacto, com intervalo de confiança.

- **Compara os grupos inteiros**, incluindo quem recusou a oferta ou não atendeu. Comparar só quem aceitou com quem não aceitou inflaria o resultado, porque quem aceita já é mais propenso a ficar.
- **`aceite_a`** é a fração do grupo A que aceitou a oferta. Ela define o custo da oferta.
- **A decisão olha o intervalo inteiro:** ampliar a ação se até o limite inferior do lucro for positivo, parar se até o limite superior for negativo, e ampliar o piloto se o intervalo cruzar o zero.

Os números do exemplo são **ilustrativos**: são os valores esperados do cenário base, não resultados medidos.

In [10]:
def impacto_piloto(n_a, churn_a, n_b, churn_b, aceite_a, mensalidade_media, nivel=0.95):
    """Impacto da ligação proativa, a partir dos resultados do piloto (grupos inteiros, sem filtrar quem aceitou)."""
    z = norm.ppf(0.5 + nivel / 2)
    queda = churn_b - churn_a
    ep = math.sqrt(churn_a * (1 - churn_a) / n_a + churn_b * (1 - churn_b) / n_b)
    v = MESES_V * mensalidade_media
    custo = n_a * C_LIGACAO + n_a * aceite_a * D

    def lucro(q):
        return n_a * q * v - custo

    ic = (queda - z * ep, queda + z * ep)
    if lucro(ic[0]) > 0:
        decisao = "ampliar a ação"
    elif lucro(ic[1]) < 0:
        decisao = "parar ou rever a oferta"
    else:
        decisao = "inconclusivo: ampliar o piloto"
    reais = lambda x: f"R$ {x:,.0f}".replace(",", ".")
    pp = lambda x: f"{100 * x:.1f}".replace(".", ",")
    return pd.Series({
        "churn evitado (p.p.)": pp(queda),
        f"IC {nivel:.0%} do churn evitado (p.p.)": f"{pp(ic[0])} a {pp(ic[1])}",
        "clientes salvos no grupo A": f"{n_a * queda:.0f}",
        "receita incremental": reais(n_a * queda * v),
        "custo de ligações e oferta": reais(custo),
        "lucro incremental": reais(lucro(queda)),
        f"IC {nivel:.0%} do lucro": f"{reais(lucro(ic[0]))} a {reais(lucro(ic[1]))}",
        "ROI": f"{lucro(queda) / custo:.1f}".replace(".", ","),
        "lucro por 1.000 ligações": reais(1000 * lucro(queda) / n_a),
        "decisão": decisao,
    })


# Exemplo ilustrativo: valores esperados do cenário base
b = piloto.set_index("cenario").loc["base"]
s = CENARIOS["base"]
alvo_base = lucro_cliente(P, V, **s) > 0
aceite_ilustrativo = (P[alvo_base] * s["da"] + (1 - P[alvo_base]) * U).mean()
print(impacto_piloto(n_a=int(b["clientes_grupo_A"]), churn_a=b["churn_com_ligacao"],
                     n_b=int(b["clientes_grupo_B"]), churn_b=b["churn_sem_acao"],
                     aceite_a=aceite_ilustrativo,
                     mensalidade_media=hist["MonthlyCharges"].values[alvo_base].mean()).to_string())

churn evitado (p.p.)                             9,3
IC 95% do churn evitado (p.p.)            2,8 a 15,9
clientes salvos no grupo A                        41
receita incremental                        R$ 39.829
custo de ligações e oferta                 R$ 11.045
lucro incremental                          R$ 28.784
IC 95% do lucro                   R$ 935 a R$ 56.632
ROI                                              2,6
lucro por 1.000 ligações                   R$ 65.269
decisão                               ampliar a ação


**Resposta:** com a função `impacto_piloto`. Quando o piloto terminar, basta trocar os cinco valores de entrada pelos medidos: o tamanho e o churn de cada grupo, o aceite no grupo A e a mensalidade média.

No exemplo ilustrativo do cenário base:

| Saída | Valor |
|---|---|
| Churn evitado | 9,3 p.p. (IC 95% de 2,8 a 15,9) |
| Clientes salvos no grupo A | 41 |
| Receita incremental | R$ 39,8 mil |
| Custo de ligações e oferta | R$ 11,0 mil |
| **Lucro incremental** | **R$ 28,8 mil** (IC 95% de R$ 0,9 mil a R$ 56,6 mil) |
| ROI | 2,6 |
| Decisão | Ampliar a ação |

Repare que o limite inferior do lucro fica pouco acima de zero. É o esperado, porque o piloto foi dimensionado no limite para detectar o efeito do cenário base. Se o efeito real for menor, a calculadora vai dizer "inconclusivo: ampliar o piloto", e não "funciona".

## Limites desta análise

- **As taxas são premissas:** `Δa`, `w` e `u` não vêm dos dados, porque não existe histórico de contatos nem de ofertas. Os lucros mostram a ordem de grandeza e a comparação entre estratégias, não um valor garantido. Medir essas taxas é a função do piloto.
- **Receita, não margem:** a TS não informou a margem, então todos os valores estão em receita.
- **O custo da oferta é um teto:** os R$ 60 supõem que o suporte grátis custa o preço de tabela. Para quem não tinha o serviço, o custo real deve ser menor.
- **O horizonte de 12 meses é premissa:** a TS não sabe a que período o `Churn` se refere.
- **A base ativa maior que o `inference` é premissa:** a TS não informou o tamanho da base.
- **`Δa` já desconta a camada reativa,** mas não sabemos quanto a camada reativa salva sozinha. Se ela salvar muito, o ganho da ligação proativa diminui.
- **A comparação com a regra favorece a regra:** ela foi escolhida olhando o `history` inteiro.
- **Não há uma base de teste intocada:** as decisões dos notebooks 06 a 09 usaram o mesmo `history`, então o desempenho real pode ser um pouco menor.